# WESLA offline trainer (Kaggle)
Settings → Accelerator: **GPU T4 x2** (or P100), Internet **on**.
Add Kaggle *Secrets*: `HF_TOKEN` (write access). Upload this repo as a Kaggle Dataset named `wesla-code`, or clone it from GitHub in the next cell.

In [ ]:
# Option A: code from a Kaggle dataset
!cp -r /kaggle/input/wesla-code /kaggle/working/wesla
# Option B: from GitHub
# !git clone https://github.com/<you>/wesla /kaggle/working/wesla
%cd /kaggle/working/wesla
!pip -q install -r requirements-train.txt

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
from huggingface_hub import login; login(os.environ['HF_TOKEN'])

## 1. Pull the queue and restore previous columns
The app pushes `train_queue.jsonl` with `python -m wesla push-queue`. Previous runs are kept in an HF dataset repo so Kaggle sessions stay stateless.

In [ ]:
QUEUE_REPO = 'your-user/wesla-train-queue'   # dataset repo
RUNS_REPO  = 'your-user/wesla-runs'          # dataset repo holding runs/wesla between sessions
from huggingface_hub import snapshot_download, HfApi
api = HfApi()
api.create_repo(RUNS_REPO, repo_type='dataset', private=True, exist_ok=True)
try:
    snapshot_download(RUNS_REPO, repo_type='dataset', local_dir='runs')
except Exception as e:
    print('no previous runs:', e)

## 2. Plan (no GPU): see how the domain clustering assigns items

In [ ]:
!python -m trainer.train --hf-queue $QUEUE_REPO --workdir runs/wesla --dry-run

## 3. Selective continual learning (LoRA columns + EWC + replay + ΔPPL gate)

In [ ]:
!python -m trainer.train --hf-queue $QUEUE_REPO --workdir runs/wesla \
    --base-model unsloth/Llama-3.2-1B-Instruct --epochs 3 --ewc-lambda 50 --replay-ratio 0.3 --delta-ppl 0.3

## 4. Baseline (base-paper style): train on every candidate, no judge/EWC/replay/gate
Upload `data/candidates.jsonl` from the app to the queue repo first.

In [ ]:
from huggingface_hub import hf_hub_download
cand = hf_hub_download(QUEUE_REPO, 'candidates.jsonl', repo_type='dataset')
!python -m trainer.train --mode train-all --candidates $cand --workdir runs/baseline \
    --base-model unsloth/Llama-3.2-1B-Instruct --epochs 3 --min-items 1

In [ ]:
!python -m eval.compare_runs runs/wesla runs/baseline --candidates $cand --markdown

## 5. Export accepted columns for Ollama (optional)

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/working/llama.cpp
!pip -q install -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
from pathlib import Path
from trainer.registry import Registry
from trainer.train import export_gguf
reg = Registry.load('runs/wesla/columns.json')
for c in reg.trained():
    print(export_gguf(c, 'unsloth/Llama-3.2-1B-Instruct', '/kaggle/working/llama.cpp', 'llama3.2:1b', Path('runs/wesla/ollama')))

## 6. Save state back to the Hub

In [ ]:
api.upload_folder(folder_path='runs', repo_id=RUNS_REPO, repo_type='dataset', commit_message='trainer run')
print('Download runs/wesla (columns.json, trained.json, ollama/) and run:  python -m wesla sync runs/wesla')